# RTI Governance Prototype: IRT (Graded Response Model) on RTI-Bench
**Project:** Data-Driven Governance Assessment Through Latent Variable Analysis of RTI Records

**Prototype pipeline:** Load data → Build ordinal indicators → Item matrix → GRM (IRT) → Latent scores → Charts

Run the cells top to bottom. **Stop after Step 2 and read the output** before moving on. The column names in RTI-Bench were not verified when this notebook was written, so Step 2 asks you to confirm them.

## Step 0: Install

In [ ]:
!pip -q install huggingface_hub girth pandas matplotlib numpy scipy

## Step 1: Load RTI-Bench
Dataset: `joyboseroy/rti-bench` on HuggingFace. The repo has **two CSV files with different columns**, so `load_dataset` fails. Load them separately:
- `hf_annotated.csv` = Source A (~1,218 cases, has `outcome`, `exemptions_cited`, `penalty_inr`, `compensation_inr`). **Used for the prototype.**
- `cic_combined.csv` = Source B (CIC PDF decisions, has many date columns such as `rti_filed_on`, `cpio_replied_on`, `date_of_decision`). Useful later for a delay/responsiveness indicator.

In [ ]:
import numpy as np, pandas as pd, re
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download

path_a = hf_hub_download("joyboseroy/rti-bench", "hf_annotated.csv", repo_type="dataset")
df = pd.read_csv(path_a)

# Source B (for later, delay indicators):
# path_b = hf_hub_download("joyboseroy/rti-bench", "cic_combined.csv", repo_type="dataset")
# df_b = pd.read_csv(path_b)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
df.head(3)

## Step 2: Confirm column mapping  *(do this by hand)*
Below, the code **guesses** which column is which, using keywords. Look at the printed columns from Step 1 and **fix the mapping** if the guess is wrong or `None`.
- `outcome`: the label (e.g. INFORMATION_DIRECTED, APPEAL_DISMISSED, PENALTY_IMPOSED, PARTIAL_RELIEF, ...)
- `exemptions`: exemption sections cited (list, string, or count)
- `penalty`: penalty amount or flag
- `compensation`: compensation amount or flag (optional)
- `authority`: public authority name (optional, for grouping later)

In [ ]:
def guess(keys, cols):
    for k in keys:                                   # exact name first
        for c in cols:
            if c.lower() == k:
                return c
    for k in keys:                                   # then keyword match
        for c in cols:
            if k in c.lower() and "adjunct" not in c.lower():
                return c
    return None

COL = {
    "outcome":      guess(["outcome", "label"], df.columns),
    "exemptions":   guess(["exemptions_cited", "exempt"], df.columns),
    "penalty":      guess(["penalty_inr", "penalt"], df.columns),
    "compensation": guess(["compensation_inr", "compens"], df.columns),
    "authority":    guess(["public_authority", "authority"], df.columns),
}
print("Guessed mapping:", COL)

# >>> If any guess is wrong or None, override here, e.g.:
# COL["outcome"] = "outcome_label"

for k, c in COL.items():
    if c:
        print(f"\n--- {k} ({c}) ---")
        print(df[c].head(5).tolist())

## Step 3: Build ordinal indicators
Each indicator is coded **0 = worse governance outcome, higher = better**, so all items point in the same direction.

| Indicator | Coding | Governance idea |
|---|---|---|
| `compliance` | 0 = CIC had to direct disclosure or impose penalty, 1 = partial relief / remanded, 2 = appeal dismissed (authority's response upheld) | Transparency / Implementation |
| `no_penalty` | 0 = penalty imposed, 1 = none | Implementation Effectiveness |
| `exemption_use` | 0 = 2+ exemptions cited, 1 = one, 2 = none | Transparency (withholding) |

**Assumption you must justify in the report:** reaching the CIC means a dispute already happened, and "appeal dismissed" is treated as the authority having acted properly. This coding is a *modelling choice*, so discuss it with Dr. Kotoky.

In [ ]:
def outcome_clean(s):
    return str(s).strip().upper().replace(" ", "_")

def count_exemptions(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return 0
    if isinstance(v, (list, tuple, np.ndarray)):
        return len(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return int(v)
    s = str(v).strip()
    if s in ("", "[]", "None", "nan", "{}"):
        return 0
    hits = re.findall(r"8\s*\(\s*1\s*\)\s*\(\s*[a-j]\s*\)", s, flags=re.I)
    return len(hits) if hits else len([x for x in re.split(r"[,;|]", s) if x.strip()])

def to_flag(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return 0
    if isinstance(v, (bool, np.bool_)):
        return int(v)
    if isinstance(v, (int, float, np.integer, np.floating)):
        return int(v > 0)
    s = str(v).strip().lower()
    if s in ("", "none", "nan", "0", "false", "no"):
        return 0
    return int(bool(re.search(r"[1-9]", s)) or s in ("true", "yes"))

def build_indicators(df, COL):
    out = pd.DataFrame(index=df.index)
    oc = df[COL["outcome"]].map(outcome_clean)
    out["outcome"] = oc
    out = out[~oc.isin(["UNKNOWN", "NAN", "NONE", ""])].copy()   # drop unlabeled cases
    oc = out["outcome"]

    comp_map = {
        "INFORMATION_DIRECTED": 0, "PENALTY_IMPOSED": 0,
        "PARTIAL_RELIEF": 1, "REMANDED": 1,
        "APPEAL_DISMISSED": 2,
    }
    out["compliance"] = oc.map(comp_map)          # other labels (withdrawn, S18, adjourned) -> NaN -> dropped

    pen = pd.Series(0, index=out.index)
    if COL["penalty"]:
        pen = df.loc[out.index, COL["penalty"]].map(to_flag)
    pen = pen | (oc == "PENALTY_IMPOSED").astype(int)
    out["no_penalty"] = 1 - pen

    if COL["exemptions"]:
        n_ex = df.loc[out.index, COL["exemptions"]].map(count_exemptions)
        out["exemption_use"] = n_ex.map(lambda n: 2 if n == 0 else (1 if n == 1 else 0))

    if COL["authority"]:
        out["authority"] = df.loc[out.index, COL["authority"]]
    return out.dropna(subset=["compliance"])

ind = build_indicators(df, COL)
ITEMS = [c for c in ["compliance", "no_penalty", "exemption_use"] if c in ind.columns]
ind[ITEMS] = ind[ITEMS].astype(int)

print("Cases kept:", len(ind), "of", len(df))
for c in ITEMS:
    print(f"\n{c}:\n", ind[c].value_counts().sort_index().to_string())

## Step 4: Item matrix + sanity checks
GRM needs **items × cases**. Before modelling, check:
- Does every item have ≥2 categories with reasonable counts?
- Are the items correlated with each other? (If correlations are ~0, there is no common latent trait to find.)
- Are they *too* correlated because they come from the same label? (Local dependence: report this honestly.)

In [ ]:
from scipy.stats import spearmanr

X = ind[ITEMS].values            # cases x items
print("Item matrix (cases x items):", X.shape)

corr = ind[ITEMS].corr(method="spearman")
print("\nSpearman correlations:\n", corr.round(2))

for c in ITEMS:
    share = ind[c].value_counts(normalize=True)
    if share.max() > 0.9:
        print(f"WARNING: '{c}' is {share.max():.0%} one category, so it will barely discriminate.")
if len(ITEMS) < 3:
    print("WARNING: fewer than 3 items. GRM will be weakly identified. Add indicators (e.g. delay, compensation).")

## Step 5: Fit the Graded Response Model (IRT)
- `Discrimination (a)`: how well an item separates high from low latent levels (higher is better).
- `Difficulty (b)`: thresholds on the latent scale where responses move up a category.

In [ ]:
from girth import grm_mml

D = ind[ITEMS].values.T.astype(int)          # items x cases
res = grm_mml(D)

a = np.asarray(res["Discrimination"])
b = np.asarray(res["Difficulty"])
params = pd.DataFrame({"item": ITEMS, "discrimination": a.round(2)})
for k in range(b.shape[1]):
    params[f"threshold_{k+1}"] = b[:, k].round(2)
params

## Step 6: Latent scores (EAP) per case

In [ ]:
def grm_eap(D, a, b, n_quad=61):
    th = np.linspace(-5, 5, n_quad)
    prior = np.exp(-0.5 * th**2); prior /= prior.sum()
    n_items, n_people = D.shape
    loglik = np.zeros((n_people, n_quad))
    for i in range(n_items):
        bi = b[i][~np.isnan(b[i])]
        cum = 1 / (1 + np.exp(-a[i] * (th[None, :] - bi[:, None])))
        cum = np.vstack([np.ones(n_quad), cum, np.zeros(n_quad)])
        probs = np.clip(cum[:-1] - cum[1:], 1e-12, 1)
        loglik += np.log(probs[D[i].astype(int)])
    post = np.exp(loglik - loglik.max(1, keepdims=True)) * prior
    post /= post.sum(1, keepdims=True)
    eap = post @ th
    sd = np.sqrt(np.maximum(post @ th**2 - eap**2, 0))
    return eap, sd

ind["theta"], ind["theta_se"] = grm_eap(D, a, b)
print(ind[["theta", "theta_se"]].describe().round(2))

## Step 7: Charts

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))

ax[0].hist(ind["theta"], bins=30)
ax[0].set_title("Latent score distribution"); ax[0].set_xlabel("theta (higher = better outcome for transparency/compliance)")

ax[1].bar(ITEMS, a)
ax[1].set_title("Item discrimination"); ax[1].set_ylabel("a")

order = ind.groupby("outcome")["theta"].mean().sort_values().index
ax[2].boxplot([ind.loc[ind.outcome == o, "theta"] for o in order], labels=order)
ax[2].set_title("Score by CIC outcome"); ax[2].tick_params(axis="x", rotation=45)

plt.tight_layout(); plt.show()

## Step 8: Summary table + what to write in the report

In [ ]:
summary = ind.groupby("outcome")["theta"].agg(["count", "mean", "std"]).round(2).sort_values("mean")
display(summary)

if "authority" in ind.columns:
    top = ind.groupby("authority")["theta"].agg(["count", "mean"]).query("count >= 5").sort_values("mean")
    print("\nAuthorities with >=5 cases (lowest and highest mean score):")
    display(pd.concat([top.head(5), top.tail(5)]).round(2))

### Limitations to state in the report (be upfront; the panel will ask)
1. **Selection bias:** RTI-Bench holds only CIC decisions, i.e. disputed cases. It says nothing about RTI applications that never reached the CIC.
2. **Few, derived indicators:** all three items come from the same rule-based extraction (label, exemption regex), so they are not independent. Part of any "latent trait" found may just be the labelling pipeline (local dependence).
3. **Label quality:** the dataset paper reports ~95% label precision on a 50-case sample and only 51% label coverage on the PDF subset.
4. **Coding is a modelling choice:** "appeal dismissed = authority acted properly" is an assumption.

### Next improvements (after the prototype works)
- Add more indicators: delay in days, compensation, first-appeal outcome (if fields exist).
- Compare with CFA (`semopy` / `factor_analyzer`) and report model fit.
- Group scores by authority or year for the final assessment.